🟢 Débutant | ⏱ 45 min | 🔑 MétéoLocale, étape 12

# Fil rouge MétéoLocale — L'historique en base `meteolocale.db`

Lundi matin chez MétéoLocale. Inès a une mauvaise nouvelle :

> « Hier, deux collègues ont modifié `meteo.csv` en même temps. L'un a écrasé le travail de l'autre. Fini les
> fichiers qui se baladent : on range tout l'historique dans une **base de données**. Et je veux pouvoir corriger
> une valeur sans tout réécrire. »

**Ce que tu as construit avant**
- Modules 01 à 07 : la fiche d'une journée, les 30 jours, la liste de dictionnaires, les fonctions.
- Module 08 : le vrai `meteo.csv`, le fichier brut nettoyé, un résumé JSON.
- Module 09 : les prévisions Open-Meteo de Toulouse.
- Module 10 : le bulletin mensuel avec pandas et deux graphiques.
- Module 11 : les classes `Releve` et `Station`, et un premier modèle de prévision.

**Ce que tu construis aujourd'hui** : la base `sorties/meteolocale.db`, recréée à neuf à chaque exécution, avec les
relevés et les prévisions, et quelques requêtes pour Inès. Les étapes s'enchaînent : exécute-les dans l'ordre.

**Comment travailler** : remplace chaque `...` par ton code (garde les noms de variables), exécute ta cellule, puis la cellule **Vérification** (`✅ OK` = réussi). Indice 💡 si tu bloques plus de 5 minutes.

**Lire un échec.** Si la vérification échoue, la dernière ligne commence par `AssertionError:`. Ça veut dire « la vérification a échoué » (ce n'est pas Python qui est cassé). Le message **en français** après les deux-points te dit quoi corriger ; les lignes en anglais au-dessus, tu peux les ignorer. Les caractères spéciaux (`°`, `—`, `«`, `î`…) : copie-colle-les depuis l'énoncé.

### Étape 1 — Créer la base et la table

**Objectif** : Ouvrir `sorties/meteolocale.db` et créer la table `releves`, à neuf.

**À faire** :
1. Ouvre la connexion dans `connexion`.
2. Supprime la table si elle existe, puis crée-la (six colonnes, comme dans le cours).
3. Valide.

**Résultat attendu** :
```
['date', 'ville', 'temp_min', 'temp_max', 'pluie_mm', 'vent_kmh']
```

<details><summary>💡 Indice</summary>

`"DROP TABLE IF EXISTS releves"`, puis le `CREATE TABLE` de la section 3 du cours.

📖 Cours, sections 2 et 3.

</details>

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

Path("sorties").mkdir(exist_ok=True)
connexion = sqlite3.connect(...)
connexion.execute(...)
connexion.execute(...)
connexion.commit()
print([c[1] for c in connexion.execute("PRAGMA table_info(releves)").fetchall()])

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/meteolocale.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "releves" in tables, "La table releves n'existe pas dans sorties/meteolocale.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
colonnes = [c[1] for c in v.execute("PRAGMA table_info(releves)").fetchall()]
v.close()
assert colonnes == ["date", "ville", "temp_min", "temp_max", "pluie_mm", "vent_kmh"], f"Colonnes attendues : date, ville, temp_min, temp_max, pluie_mm, vent_kmh. Tu as {colonnes}."
print("✅ OK")

### Étape 2 — Ranger les 180 relevés

**Objectif** : Insérer tout `meteo.csv`, avec une requête paramétrée.

**À faire** :
1. Une boucle d'`INSERT` sur `to_dict("records")`, puis un `commit`.
2. Range dans `nb_releves` le nombre de lignes en base.

**Résultat attendu** :
```
180
```

<details><summary>💡 Indice</summary>

Dans la boucle : `connexion.execute("INSERT INTO releves VALUES (?, ?, ?, ?, ?, ?)", (f["date"], f["ville"], f["temp_min"], f["temp_max"], f["pluie_mm"], f["vent_kmh"]))`.

📖 Cours, section 4.

</details>

In [ ]:
for f in pd.read_csv("../donnees/meteo.csv").to_dict("records"):
    ...
connexion.commit()
nb_releves = len(connexion.execute("SELECT * FROM releves").fetchall())
print(nb_releves)

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/meteolocale.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "releves" in tables, "La table releves n'existe pas dans sorties/meteolocale.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
n = v.execute("SELECT COUNT(*) FROM releves").fetchall()[0][0]
v.close()
assert n == 180, f"La base doit contenir 180 relevés (validés), elle en contient {n}."
print("✅ OK")

### Étape 3 — La table des prévisions

**Objectif** : Ajouter une table `previsions` (date, temp_min, temp_max, pluie_mm) remplie avec le JSON d'Open-Meteo (module 09).

**À faire** :
1. Crée la table à neuf (`DROP` + `CREATE`).
2. Pour chaque position `i`, insère `(jours[i], mini[i], maxi[i], pluies[i])`.
3. Valide, puis range dans `nb_previsions` le nombre de lignes.

**Résultat attendu** :
```
7
```

<details><summary>💡 Indice</summary>

Les `...` remplacent seulement ce qui va entre les parenthèses de `connexion.execute`. Les deux lignes complètes : `connexion.execute("CREATE TABLE previsions (date TEXT, temp_min REAL, temp_max REAL, pluie_mm REAL)")` et, dans la boucle, `connexion.execute("INSERT INTO previsions VALUES (?, ?, ?, ?)", (jours[i], mini[i], maxi[i], pluies[i]))`. Si tu vois `execute() argument 1 must be str, not sqlite3.Cursor`, tu as mis un `execute` dans un `execute`.

📖 Cours, sections 3 et 4 ; module 09 (JSON).

</details>

In [ ]:
import json

with open("../donnees/open_meteo_toulouse.json", encoding="utf-8") as fichier:
    donnees = json.load(fichier)
jours = donnees["daily"]["time"]
mini = donnees["daily"]["temperature_2m_min"]
maxi = donnees["daily"]["temperature_2m_max"]
pluies = donnees["daily"]["precipitation_sum"]

connexion.execute("DROP TABLE IF EXISTS previsions")
connexion.execute(...)
for i in range(len(jours)):
    connexion.execute(...)
connexion.commit()
nb_previsions = len(connexion.execute("SELECT * FROM previsions").fetchall())
print(nb_previsions)

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/meteolocale.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "previsions" in tables, "La table previsions n'existe pas dans sorties/meteolocale.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
lignes = v.execute("SELECT * FROM previsions").fetchall()
v.close()
assert len(lignes) == 7, f"La table previsions doit contenir 7 lignes validées, elle en contient {len(lignes)}."
assert lignes[3] == ("2026-10-04", 9.1, 18.2, 12.6), f"La 4e prévision doit être ('2026-10-04', 9.1, 18.2, 12.6), elle est {lignes[3]}."
print("✅ OK")

### Étape 4 — Corriger et nettoyer

**Objectif** : Appliquer deux demandes des stations, sans réécrire de fichier.

**À faire** :
1. Perpignan signale : le 2026-09-14, la pluie était de 21.9 mm (pas 19.9). Corrige avec `UPDATE`.
2. Un collègue a inséré un relevé de test (ville `"Test"`) : supprime-le avec `DELETE`.
3. Valide une seule fois, à la fin.

**Résultat attendu** :
```
[(21.9,)] 180
```

<details><summary>💡 Indice</summary>

`UPDATE releves SET pluie_mm = ? WHERE ville = ? AND date = ?` avec `(21.9, "Perpignan", "2026-09-14")` ; `DELETE FROM releves WHERE ville = ?` avec `("Test",)`.

📖 Cours, sections 6 et 7.

</details>

In [ ]:
connexion.execute("INSERT INTO releves VALUES (?, ?, ?, ?, ?, ?)", ("2026-09-15", "Test", 0.0, 0.0, 0.0, 0))
connexion.commit()

connexion.execute(...)
connexion.execute(...)
connexion.commit()
print(connexion.execute("SELECT pluie_mm FROM releves WHERE ville = ? AND date = ?", ("Perpignan", "2026-09-14")).fetchall(),
      len(connexion.execute("SELECT * FROM releves").fetchall()))

In [ ]:
# Vérification
import sqlite3
from pathlib import Path
Path("sorties").mkdir(exist_ok=True)
v = sqlite3.connect("sorties/meteolocale.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "releves" in tables, "La table releves n'existe pas dans sorties/meteolocale.db : exécute ta cellule jusqu'au bout (CREATE TABLE, puis commit)."
case = v.execute("SELECT pluie_mm FROM releves WHERE ville = 'Perpignan' AND date = '2026-09-14'").fetchall()
test = v.execute("SELECT COUNT(*) FROM releves WHERE ville = 'Test'").fetchall()[0][0]
n = v.execute("SELECT COUNT(*) FROM releves").fetchall()[0][0]
nb_219 = v.execute("SELECT COUNT(*) FROM releves WHERE pluie_mm = 21.9").fetchall()[0][0]
v.close()
assert case == [(21.9,)], f"La pluie de Perpignan le 2026-09-14 vaut {case} pour une nouvelle connexion : UPDATE … WHERE ville = ? AND date = ?, puis commit."
assert nb_219 == 1, f"{nb_219} lignes valent 21.9 mm : ton UPDATE doit viser une seule ligne. Recommence depuis l'étape 1."
assert test == 0, "Le relevé Test est encore là : DELETE FROM releves WHERE ville = ?"
assert n == 180, f"La base doit contenir 180 relevés, elle en contient {n}. Si c'est plus, recommence depuis l'étape 1."
print("✅ OK")

### Étape 5 — Les grosses pluies de Toulouse

**Objectif** : Première requête pour Inès : les jours de Toulouse avec plus de 10 mm, du plus arrosé au moins arrosé.

**À faire** :
1. Range dans `grosses_pluies` la liste des `(date, pluie_mm)`.
2. Affiche une ligne par jour avec une boucle et un déballage.

**Résultat attendu** :
```
2026-09-01 : 17.4 mm
2026-09-17 : 16.1 mm
2026-09-29 : 11.6 mm
2026-09-30 : 11.4 mm
2026-09-19 : 10.8 mm
2026-09-07 : 10.7 mm
2026-09-04 : 10.4 mm
```

<details><summary>💡 Indice</summary>

`"SELECT date, pluie_mm FROM releves WHERE ville = ? AND pluie_mm > ? ORDER BY pluie_mm DESC"` et `("Toulouse", 10)`.

📖 Cours, section 5.

</details>

In [ ]:
grosses_pluies = connexion.execute(..., ...).fetchall()
for date, pluie in grosses_pluies:
    print(f"{date} : {pluie} mm")

In [ ]:
# Vérification
assert "grosses_pluies" in dir(), "La variable grosses_pluies n'existe pas : ta cellule a planté (lis la dernière ligne du message)."
assert len(grosses_pluies) == 7, f"Il y a 7 jours à plus de 10 mm à Toulouse, tu en as {len(grosses_pluies)}."
assert grosses_pluies[0] == ('2026-09-01', 17.4), f"La première ligne doit être le jour le plus arrosé, ('2026-09-01', 17.4) : ORDER BY pluie_mm DESC."
print("✅ OK")

### Étape 6 — Le tableau par ville, depuis la base

**Objectif** : Lire la base avec pandas et refaire le tableau du bulletin (module 10), avec la valeur corrigée de Perpignan.

**À faire** :
1. Range dans `historique` toute la table `releves` (`pd.read_sql`).
2. Range dans `tableau` la moyenne de `temp_max` par ville (doubles crochets, comme au module 10), puis ajoute la colonne `pluie_totale` : le total de `pluie_mm` par ville. Arrondis à 1 chiffre.

**Résultat attendu** :
```
Perpignan : pluie_totale = 138.3 (avec la correction de l'étape 4)
```

<details><summary>💡 Indice</summary>

`historique.groupby("ville")["pluie_mm"].sum().round(1)`.

📖 Cours, section 8 ; module 10, section 8.

</details>

In [ ]:
historique = pd.read_sql(..., connexion)
tableau = historique.groupby("ville")[["temp_max"]].mean().round(1)
tableau["pluie_totale"] = ...
tableau

In [ ]:
# Vérification
assert "tableau" in dir() and "historique" in dir(), "Exécute d'abord ta cellule."
assert historique is not ... and len(historique) == 180, "historique doit contenir les 180 relevés : pd.read_sql(\"SELECT * FROM releves\", connexion)."
assert "pluie_totale" in tableau.columns, "Ajoute la colonne pluie_totale au tableau."
assert tableau["pluie_totale"]["Perpignan"] == 138.3, f"La pluie totale de Perpignan doit valoir 138.3 (avec la correction de l'étape 4), tu as {tableau['pluie_totale']['Perpignan']}."
print("✅ OK")

### Étape 7 — Une fonction pour Inès, et on raccroche

**Objectif** : Écrire `historique_ville(chemin_base, ville)` : elle ouvre la base, lit les relevés de la ville dans un DataFrame (requête paramétrée), **ferme** la connexion et renvoie le DataFrame.

**À faire** :
1. Ferme d'abord la connexion des étapes précédentes : `connexion.close()`.
2. Écris la fonction, puis range dans `albi` le résultat pour Albi.
3. Range dans `pluie_albi` la pluie totale d'Albi, arrondie à 1 chiffre.

**Résultat attendu** :
```
30 125.7
```

<details><summary>💡 Indice</summary>

Trois lignes et un `return` : `sqlite3.connect(chemin_base)`, `pd.read_sql("SELECT * FROM releves WHERE ville = ?", base, params=(ville,))`, `base.close()`.

📖 Cours, section 8 ; module 07.

</details>

In [ ]:
connexion.close()


def historique_ville(chemin_base, ville):
    ...


albi = historique_ville("sorties/meteolocale.db", "Albi")
pluie_albi = round(albi["pluie_mm"].sum(), 1)
print(len(albi), pluie_albi)

In [ ]:
# Vérification
assert "historique_ville" in dir(), "La fonction historique_ville n'existe pas."
assert "pluie_albi" in dir(), "La variable pluie_albi n'existe pas : ta cellule a planté. Ta fonction renvoie-t-elle bien le DataFrame (return) ?"
assert len(albi) == 30, f"albi doit contenir 30 lignes, il en contient {len(albi)}."
assert pluie_albi == 125.7, f"pluie_albi doit valoir 125.7, tu as {pluie_albi}."
assert len(historique_ville("sorties/meteolocale.db", "Tarbes")) == 30, "La fonction doit utiliser son paramètre ville (avec un ?), pas \"Albi\" écrit en dur."
print("✅ OK")

## 🏁 Ce que tu as construit

La base `sorties/meteolocale.db`, recréée à neuf à chaque exécution, avec :
- la table `releves` : les 180 relevés de septembre, corrigés **sans réécrire de fichier** ;
- la table `previsions` : les 7 jours d'Open-Meteo ;
- des requêtes **paramétrées**, sûres même avec une apostrophe, et une fonction `historique_ville` pour Inès.

**Au module 13**, le projet final : tu assembles tout. Fichier brut → nettoyage → base SQLite → statistiques →
graphique → bulletin. Le pipeline complet de MétéoLocale.